### Botlikh morphological transducer: analyses of the corpus words

This notebook takes every word of the table `evaluation/corpus/botlikh_corpus_words.xlsx` (one row per word occurrence of the Spoken corpus of Botlikh) and collects all analyses that the transducer gives for it. It produces

1. tables for the manual evaluation of the analyses;
2. a comparison of the transducer with and without clitics;
3. statistics by part of speech: how many words of every part of speech are recognised, how many analyses they get and which words are not recognised.

**How to run.** Open the notebook in Google Colab and choose *Runtime → Run all*. The notebook installs `lexd` and HFST, takes the transducer, compiles it from the sources and then does the evaluation. Nothing in the notebook is random: the same sources and the same input file always give the same output. Checksums of the sources, of the input file and of the results are printed, so two runs can be compared directly.

#### 1. Settings

* `GITHUB_URL`: the repository with the transducer and the evaluation data.
* `GITHUB_COMMIT`: an optional commit hash that pins the exact version of the transducer.
* `PROJECT_DIR`: the folder of the repository (with the `Makefile` of the transducer).
* `OUT_DIR`: the folder for all tables and figures.
* `VARIANTS`: the two versions of the transducer that are compared. In the first one no clitics are allowed; in the second one a chain of up to three clitics can follow a word of any part of speech.
* `TABLE_ANALYZER`: the analyzer for the tables of the manual evaluation: the version with clitics after all words.


In [ ]:
GITHUB_URL = 'https://github.com/mbogautdinov/botlikh_transducer'
GITHUB_COMMIT = ''

PROJECT_DIR = 'botlikh_transducer'
OUT_DIR = 'eval_output'

VARIANTS = {
    'no clitics': 'bot_noclitics_analyzer',
    'all clitics': 'bot_allclitics_analyzer',
}

TABLE_ANALYZER = 'bot_allclitics_analyzer'

ANALYZERS = sorted(set(VARIANTS.values()) | {TABLE_ANALYZER})


#### 2. Tools

The transducer is compiled with `lexd` and HFST. If they are not installed yet, the cell installs them: first from the standard Ubuntu packages and, if `lexd` is not found there, from the package repository of the Apertium project. The helper `sh` runs a shell command, prints the end of its output and stops the notebook if the command fails. The versions of the tools are printed at the end, so that they are saved together with the results.


In [ ]:
import os
import shutil
import subprocess


def sh(cmd, check=True, tail=3000):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout[-tail:])
    if check and r.returncode != 0:
        raise RuntimeError('command failed: ' + cmd)


TOOLS = ('lexd', 'hfst-twolc', 'hfst-compose-intersect', 'hfst-lookup', 'make')

if any(shutil.which(t) is None for t in TOOLS):
    sh('apt-get update -qq && apt-get install -y -qq hfst make', check=False)
    if shutil.which('lexd') is None:
        sh('apt-get install -y -qq lexd', check=False)
    if shutil.which('lexd') is None or shutil.which('hfst-twolc') is None:
        sh('wget -q https://apertium.projectjj.com/apt/install-nightly.sh -O /tmp/install-nightly.sh'
           ' && bash /tmp/install-nightly.sh && apt-get install -y -qq hfst lexd')

missing = [t for t in TOOLS if shutil.which(t) is None]
assert not missing, 'not installed: ' + ', '.join(missing)

sh('lexd --version 2>&1 | head -1; hfst-lookup --version 2>&1 | head -1; make --version | head -1; python3 --version')


#### 3. The transducer

The repository is cloned from GitHub. If the folder `PROJECT_DIR` already exists (for example, a local clone), it is used as it is. The commit of the repository is printed, so that the results can be traced to an exact version of the transducer.


In [ ]:
if not os.path.isdir(PROJECT_DIR):
    assert GITHUB_URL, 'set GITHUB_URL in the settings'
    sh('git clone -q ' + GITHUB_URL + ' ' + PROJECT_DIR)
    if GITHUB_COMMIT:
        sh('git -C ' + PROJECT_DIR + ' checkout -q ' + GITHUB_COMMIT)

assert os.path.isfile(os.path.join(PROJECT_DIR, 'Makefile')), 'the folder with the transducer was not found'
os.makedirs(OUT_DIR, exist_ok=True)
print('project:', os.path.abspath(PROJECT_DIR))
sh('git -C ' + PROJECT_DIR + ' log -1 --format="commit %H (%ad)" --date=short', check=False)


The transducer is rebuilt from scratch from the `lexd` and `twol` sources. `make` builds the analyzers, among them the two versions that are compared here. No Python scripts are called at this step: the lexicons are a part of the repository.


In [ ]:
sh('make -C ' + PROJECT_DIR + ' clean')
sh('make -C ' + PROJECT_DIR, tail=600)
sh('ls -la ' + ' '.join(os.path.join(PROJECT_DIR, a + '.hfstol') for a in ANALYZERS))


The cell below prints one checksum for all sources of the transducer (the `lexd` and `twol` files and the `Makefile`) and one for the input data. If these two values are the same on two computers, all results below must be the same as well.


In [ ]:
DATA = os.path.join(PROJECT_DIR, 'evaluation', 'corpus', 'botlikh_corpus_words.xlsx')
import glob
import hashlib


def sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


src_files = sorted(
    glob.glob(os.path.join(PROJECT_DIR, '*', '*.lexd'))
    + glob.glob(os.path.join(PROJECT_DIR, '*', '*.twol'))
    + glob.glob(os.path.join(PROJECT_DIR, '*.twol'))
    + [os.path.join(PROJECT_DIR, 'Makefile')]
)
h = hashlib.sha256()
for p in src_files:
    h.update(os.path.relpath(p, PROJECT_DIR).encode('utf-8'))
    with open(p, 'rb') as f:
        h.update(f.read())

print('transducer sources:', len(src_files), 'files, sha256', h.hexdigest())
print('input data:        ', os.path.relpath(DATA, PROJECT_DIR), 'sha256', sha256(DATA))


The function `lookup` sends a list of word forms to `hfst-lookup` and returns a dictionary `{form: [analyses]}`. An empty list means that the form is not recognised. The analyses of a form are sorted, so the order does not depend on the version of HFST.

Two more helpers read an analysis: `proposed_pos` returns the parts of speech that the analyses of a form propose (the first tag in capitals, such as `N` or `ADV`), and `clitic_tags` returns the tags of the clitics (the tags after `=`). A clitic that is written as a separate word has an analysis without a stem (`=<quot>`); it is counted as `CLITIC`.


In [ ]:
import re


def lookup(analyzer, forms):
    forms = [f for f in dict.fromkeys(forms) if f and not any(ch.isspace() for ch in f)]
    path = os.path.join(PROJECT_DIR, analyzer + '.hfstol')
    r = subprocess.run(['hfst-lookup', '-q', path], input='\n'.join(forms) + '\n',
                       capture_output=True, text=True, check=True)
    res = {f: [] for f in forms}
    for line in r.stdout.splitlines():
        parts = line.split('\t')
        if len(parts) >= 2 and parts[0] in res and not parts[1].endswith('+?'):
            if parts[1] not in res[parts[0]]:
                res[parts[0]].append(parts[1])
    return {f: sorted(a) for f, a in res.items()}


POS_TAG = re.compile(r'<([A-Z]+)>')
CLITIC_TAG = re.compile(r'=<([^>]+)>')


def proposed_pos(analyses):
    found = set()
    for a in analyses:
        m = POS_TAG.search(a)
        if m:
            found.add(m.group(1))
        elif a.startswith('='):
            found.add('CLITIC')
    return sorted(found)


def clitic_tags(analyses):
    return sorted({t for a in analyses for t in CLITIC_TAG.findall(a)})


for name, analyzer in VARIANTS.items():
    print(name, lookup(analyzer, ['хъуча', 'хъучиди', 'хъучала', 'гьикьила']))


#### 4. Figures: common style

All figures use the same two colours: blue for the transducer without clitics and orange for the transducer with clitics after all words. The pair was checked for the common forms of colour blindness. Grey is used for context, such as the total number of words. Values are also printed in tables next to every figure, so no number has to be read from a colour alone.

`grouped_bars` draws one group of bars per row of a table and one bar per transducer; `finish` adds the title, the axis label and the legend and removes everything that is not data.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

COLORS = {'no clitics': '#2a78d6', 'all clitics': '#eb6834'}
GREY = '#c3c2b7'
INK = '#0b0b0b'
INK2 = '#52514e'
MUTED = '#898781'
GRID = '#e1e0d9'
SURFACE = '#fcfcfb'
BLUES = LinearSegmentedColormap.from_list(
    'blues', [SURFACE, '#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.edgecolor': GREY, 'axes.labelcolor': INK2, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.dpi': 110,
})


def finish(ax, title, label='', horizontal=False, legend=False):
    ax.set_title(title, pad=10)
    ax.set_axisbelow(True)
    if horizontal:
        ax.set_xlabel(label)
        ax.grid(axis='x', color=GRID, linewidth=0.8)
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='both', length=0)
    else:
        ax.set_ylabel(label)
        ax.grid(axis='y', color=GRID, linewidth=0.8)
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='both', length=0)
    if legend:
        ax.legend(loc='best', handlelength=1.2, handleheight=1.0, borderaxespad=0.2)


def grouped_bars(ax, table, horizontal=False, fmt='{:.0f}', labels=True, total=None):
    groups = list(table.index)
    series = list(table.columns)
    pos = np.arange(len(groups))
    width = 0.8 / len(series)
    if total is not None:
        if horizontal:
            ax.barh(pos, total.values, height=0.86, color=GRID, label='all words', zorder=1)
        else:
            ax.bar(pos, total.values, width=0.86, color=GRID, label='all words', zorder=1)
    for i, s in enumerate(series):
        offset = (i - (len(series) - 1) / 2) * width
        values = table[s].values
        if horizontal:
            bars = ax.barh(pos + offset, values, height=width, color=COLORS[s], edgecolor=SURFACE,
                           linewidth=1.2, label=s, zorder=2)
        else:
            bars = ax.bar(pos + offset, values, width=width, color=COLORS[s], edgecolor=SURFACE,
                          linewidth=1.2, label=s, zorder=2)
        if labels:
            ax.bar_label(bars, labels=[fmt.format(v) for v in values], padding=2, fontsize=8, color=INK2)
    if horizontal:
        ax.set_yticks(pos)
        ax.set_yticklabels(groups)
        ax.invert_yaxis()
        ax.margins(x=0.12)
    else:
        ax.set_xticks(pos)
        ax.set_xticklabels(groups)
        ax.margins(y=0.12)


def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(OUT_DIR, name + '.' + ext), dpi=200, bbox_inches='tight')


#### 5. The words of the corpus

The table has Russian column names; they are replaced by English ones (`COLUMNS`). The word forms are taken from the column `word`. Preprocessing: punctuation marks at the edges of the word are removed (three words of the table end in a full stop or a comma) and the first letter is made lower-case. The rest of the word is not changed, so the palochka stays as it is written in the corpus. The result is the column `form`.

Some rows are left out of all statistics; the reason is written in the column `excluded`:

* `not Botlikh`: Russian and Avar words (column `language`);
* `empty row`: rows without a gloss and without a part of speech (placeholders such as `<?>` or `RUSSIAN`, where the alignment of the corpus failed);
* `unclear gloss`: words with a question mark in the gloss, that is, words that the annotators of the corpus could not gloss with certainty.

These rows stay in the tables for the manual evaluation, but the statistics below are computed for the remaining Botlikh words (`botlikh`). The column `pos` is the part of speech that was assigned to the word by hand; an empty value is shown as `not marked`.


In [ ]:
import pandas as pd

COLUMNS = {
    'Язык слова': 'language',
    'Слово': 'word',
    'Сегментация': 'segmentation',
    'Глосса': 'gloss',
    'Часть речи': 'pos',
    'Предложение (ботлихский)': 'sentence',
    'Предложение (русский)': 'translation_ru',
    'Предложение (английский)': 'translation_en',
    'Документ': 'document',
    'Говорящий': 'speaker',
    '№ предложения': 'sentence_nr',
    '№ слова': 'word_nr',
    'Примечание': 'note',
}


PUNCTUATION = ' .,;:!?…'


def prepare(word):
    w = str(word).strip(PUNCTUATION)
    return w[:1].lower() + w[1:]


words = pd.read_excel(DATA).rename(columns=COLUMNS)

words['excluded'] = ''
words.loc[words['language'] != 'Botlikh', 'excluded'] = 'not Botlikh'
words.loc[(words['excluded'] == '') & words['gloss'].isna() & words['pos'].isna(), 'excluded'] = 'empty row'
words.loc[(words['excluded'] == '') & words['gloss'].fillna('').str.contains('?', regex=False), 'excluded'] = 'unclear gloss'

words['pos'] = words['pos'].fillna('not marked')
words['form'] = words['word'].map(prepare)

print(len(words), 'rows;', words['language'].value_counts().to_dict())
print('left out:', words.loc[words['excluded'] != '', 'excluded'].value_counts().to_dict())
print('Botlikh words in the statistics:', int((words['excluded'] == '').sum()))
words[['word', 'form', 'segmentation', 'gloss', 'pos']].head(10)


#### 6. Analyses

Every form is analysed with all analyzers. For each of the two compared versions the table gets three columns: the number of analyses, the proposed parts of speech and the tags of the clitics. The full lists of analyses are kept in the dictionary `results`.


In [ ]:
results = {a: lookup(a, words['form'].tolist()) for a in ANALYZERS}

for name, analyzer in VARIANTS.items():
    res = results[analyzer]
    words['n (' + name + ')'] = words['form'].map(lambda f: len(res.get(f, [])))
    words['pos (' + name + ')'] = words['form'].map(lambda f: proposed_pos(res.get(f, [])))
    words['clitics (' + name + ')'] = words['form'].map(lambda f: clitic_tags(res.get(f, [])))

botlikh = words[words['excluded'] == ''].copy()
print(len(botlikh), 'Botlikh tokens,', botlikh['form'].nunique(), 'types')


#### 7. Tables for the manual evaluation

Four tables are made for the analyzer `TABLE_ANALYZER`. The table to fill in by hand is `annotation`:

* `annotation`: one row per analysis of a word in a given reading. Occurrences with the same form, segmentation, gloss and part of speech are merged into one row (`frequency` is their number; the sentence is the first one), because the same analysis of the same word in the same reading is either right or wrong everywhere. Only the words that are counted in the statistics and have at least one analysis are listed. The column `correct` is empty: `1` for an analysis that agrees with the gloss of the corpus, `0` for a wrong one; `comment` is free;
* `words`: one row per word occurrence, in the order of the input file, with all analyses in one cell, two empty columns (`evaluation`, `comment`) and the column `excluded` (empty for the words that are counted in the statistics);
* `analyses`: one row per analysis, with an empty column `correct`; a word without analyses gets one row with an empty analysis, and `row` is the number of the row in the input file;
* `types`: one row per different word form with its frequency.

In these tables the part of speech is left as it is in the input file (an empty cell is not replaced by `not marked`).


In [ ]:
res = results[TABLE_ANALYZER]

source = words.assign(pos=words['pos'].replace('not marked', ''))

table_words = source[list(COLUMNS.values()) + ['form', 'excluded']].copy()
table_words['n_analyses'] = table_words['form'].map(lambda f: len(res.get(f, [])))
table_words['analyses'] = table_words['form'].map(lambda f: '\n'.join(res.get(f, [])))
table_words['evaluation'] = ''
table_words['comment'] = ''
first = ['language', 'word', 'form', 'segmentation', 'gloss', 'pos', 'n_analyses', 'analyses', 'evaluation', 'comment', 'excluded']
table_words = table_words[first + [c for c in table_words.columns if c not in first]]

rows = []
for i, r in source.iterrows():
    for a in (res.get(r['form'], []) or ['']):
        rows.append({
            'row': i + 2, 'language': r['language'], 'word': r['word'], 'form': r['form'],
            'segmentation': r['segmentation'], 'gloss': r['gloss'], 'pos': r['pos'],
            'analysis': a, 'correct': '', 'comment': '', 'excluded': r['excluded'],
            'sentence': r['sentence'], 'translation_ru': r['translation_ru'],
        })
table_analyses = pd.DataFrame(rows)

table_types = (
    source.groupby(['language', 'form'], sort=True)
    .agg(frequency=('word', 'size'),
         pos=('pos', lambda s: ', '.join(sorted(set(x for x in s if x)))),
         glosses=('gloss', lambda s: ' | '.join(sorted(set(s.dropna().astype(str))))))
    .reset_index()
)
table_types['n_analyses'] = table_types['form'].map(lambda f: len(res.get(f, [])))
table_types['analyses'] = table_types['form'].map(lambda f: '\n'.join(res.get(f, [])))

reading = ['pos', 'form', 'segmentation', 'gloss']
counted = table_analyses[(table_analyses['excluded'] == '') & (table_analyses['analysis'] != '')].copy()
counted[reading] = counted[reading].fillna('')
table_annotation = (
    counted.groupby(reading + ['analysis'], sort=True)
    .agg(frequency=('row', 'size'), row=('row', 'min'), sentence=('sentence', 'first'), translation_ru=('translation_ru', 'first'))
    .reset_index()
)
pos_order = {p: k for k, p in enumerate(counted['pos'].value_counts().index)}
table_annotation = table_annotation.sort_values(
    ['pos', 'form', 'gloss', 'analysis'], key=lambda c: c.map(pos_order) if c.name == 'pos' else c).reset_index(drop=True)
table_annotation.insert(0, 'id', range(1, len(table_annotation) + 1))
table_annotation['correct'] = ''
table_annotation['comment'] = ''
table_annotation = table_annotation[['id', 'pos', 'form', 'segmentation', 'gloss', 'analysis', 'correct', 'comment',
                                     'frequency', 'row', 'sentence', 'translation_ru']]

print('analyzer:', TABLE_ANALYZER)
print(len(table_words), 'word occurrences,', len(table_types), 'different forms,', len(table_analyses), 'rows in the table of analyses')
print(len(table_annotation), 'rows to annotate;', table_annotation.groupby('pos', sort=False).size().to_dict())
table_annotation.head(10)


#### 8. With and without clitics: overview

The table gives the main numbers for the Botlikh words that are counted in the statistics: the number of tokens and types, how many of them are recognised, and how many analyses a recognised token gets on average. Coverage only says that a word gets some analysis; whether the analysis is right is the subject of the manual evaluation.


In [ ]:
def metrics(df, label, names=None):
    out = []
    types = df.drop_duplicates('form')
    for name in (names or VARIANTS):
        n, nt = df['n (' + name + ')'], types['n (' + name + ')']
        out.append({
            'data': label, 'transducer': name,
            'tokens': len(df), 'recognised tokens': int((n > 0).sum()), 'token coverage': round((n > 0).mean(), 4),
            'types': len(types), 'recognised types': int((nt > 0).sum()), 'type coverage': round((nt > 0).mean(), 4),
            'analyses': int(n.sum()),
            'analyses per recognised token': round(n[n > 0].mean(), 2) if (n > 0).any() else 0,
        })
    return out


metrics_table = pd.DataFrame(metrics(botlikh, 'Botlikh words of the corpus'))
metrics_table


The overview figure is the same in both notebooks, so it is defined once as a function. It has four panels:

* **coverage**: the share of tokens and of types that get at least one analysis;
* **number of analyses per token**: how many tokens get no analysis, one analysis, two analyses and so on (the mean for the recognised tokens is given in the legend);
* **proposed parts of speech**: for every part of speech, the number of tokens that have at least one analysis with this part of speech (a token with several analyses can be counted for several parts of speech);
* **coverage in the parts of the data**: the same coverage of tokens, separately for every text or for every part of speech of the corpus.


In [ ]:
def count_table(df, key_col):
    out = {}
    for name in VARIANTS:
        counts = {}
        for keys in df[key_col + ' (' + name + ')']:
            for k in keys:
                counts[k] = counts.get(k, 0) + 1
        out[name] = counts
    table = pd.DataFrame(out).fillna(0).astype(int).sort_index()
    return table.loc[table.sum(axis=1).sort_values(ascending=False, kind='stable').index]


def overview_figure(df, group_col, group_title, name):
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8), constrained_layout=True)
    types = df.drop_duplicates('form')

    coverage = pd.DataFrame(
        {v: [100 * (df['n (' + v + ')'] > 0).mean(), 100 * (types['n (' + v + ')'] > 0).mean()] for v in VARIANTS},
        index=['tokens', 'types'])
    grouped_bars(axes[0, 0], coverage, fmt='{:.1f}')
    axes[0, 0].set_ylim(0, 100)
    finish(axes[0, 0], 'Coverage', '% with at least one analysis', legend=True)

    bins = ['0', '1', '2', '3', '4', '5+']
    dist = pd.DataFrame(
        {v: df['n (' + v + ')'].clip(upper=5).value_counts().reindex(range(6), fill_value=0).values for v in VARIANTS},
        index=bins)
    grouped_bars(axes[0, 1], dist)
    finish(axes[0, 1], 'Number of analyses per token', 'tokens')
    means = {v: df.loc[df['n (' + v + ')'] > 0, 'n (' + v + ')'].mean() for v in VARIANTS}
    handles, _ = axes[0, 1].get_legend_handles_labels()
    axes[0, 1].legend(handles, [v + ': mean ' + format(means[v], '.2f') + ' per recognised token' for v in VARIANTS],
                      loc='upper right', handlelength=1.2)
    axes[0, 1].set_xlabel('analyses')

    pos = count_table(df, 'pos')
    grouped_bars(axes[1, 0], pos, horizontal=True)
    finish(axes[1, 0], 'Proposed parts of speech', 'tokens with at least one such analysis', horizontal=True)

    by_group = pd.DataFrame(
        {v: 100 * df.groupby(group_col, sort=False)['n (' + v + ')'].apply(lambda s: (s > 0).mean()) for v in VARIANTS})
    sizes = df.groupby(group_col, sort=False).size()
    by_group = by_group.loc[sizes.sort_values(ascending=False).index]
    by_group.index = [str(g) + ' (' + str(sizes[g]) + ')' for g in by_group.index]
    grouped_bars(axes[1, 1], by_group, horizontal=True, fmt='{:.0f}')
    axes[1, 1].set_xlim(0, 100)
    finish(axes[1, 1], group_title, '% of tokens with at least one analysis', horizontal=True)

    save_figure(fig, name)
    plt.show()
    return coverage.round(2), dist, pos, by_group.round(1)


In this notebook the fourth panel shows the coverage for every part of speech of the corpus (the number of tokens is given in brackets). Verbs are not implemented in the transducer, so the analyses that verb forms get belong to other parts of speech and have to be checked by hand.


In [ ]:
coverage, distribution, proposed, coverage_by_pos = overview_figure(
    botlikh, 'pos', 'Coverage by part of speech of the corpus', 'corpus_fig_overview')


#### 9. With and without clitics: what the clitics add

The figure has two panels. The first one shows which clitics the transducer finds: for every clitic tag, the number of tokens that have at least one analysis with this clitic (several clitics have the same form, so one token is often counted for several tags). The second panel splits the tokens into three groups: recognised by both versions, recognised only when clitics are allowed, and not recognised at all. The second group is the gain in coverage.


In [ ]:
def clitics_figure(df, name):
    a, b = 'n (no clitics)', 'n (all clitics)'
    clitics = count_table(df, 'clitics')['all clitics']
    clitics = clitics[clitics > 0].head(15)
    groups = pd.Series({
        'recognised by both': int((df[a] > 0).sum()),
        'only with clitics': int(((df[a] == 0) & (df[b] > 0)).sum()),
        'not recognised': int((df[b] == 0).sum()),
    })

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True, gridspec_kw={'width_ratios': [1.15, 1]})

    bars = axes[0].barh(range(len(clitics)), clitics.values, height=0.62, color=COLORS['all clitics'],
                        edgecolor=SURFACE, linewidth=1.2)
    axes[0].bar_label(bars, padding=2, fontsize=8, color=INK2)
    axes[0].set_yticks(range(len(clitics)))
    axes[0].set_yticklabels(clitics.index)
    axes[0].invert_yaxis()
    axes[0].margins(x=0.1)
    finish(axes[0], 'Clitics found by the transducer with clitics', 'tokens with at least one such analysis', horizontal=True)

    group_colors = [COLORS['no clitics'], COLORS['all clitics'], GREY]
    bars = axes[1].bar(range(3), groups.values, width=0.5, color=group_colors, edgecolor=SURFACE, linewidth=1.2)
    axes[1].bar_label(bars, labels=[str(v) + ' (' + format(100 * v / len(df), '.1f') + '%)' for v in groups.values],
                      padding=2, fontsize=9, color=INK2)
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(groups.index)
    axes[1].margins(y=0.12)
    finish(axes[1], 'Recognised with and without clitics', 'tokens')

    save_figure(fig, name)
    plt.show()
    return clitics.rename('tokens').rename_axis('clitic').reset_index(), groups.rename('tokens').rename_axis('group').reset_index()


clitics, groups = clitics_figure(botlikh, 'corpus_fig_clitics')


The forms that are recognised only with clitics, with their part of speech and gloss in the corpus. The gloss shows whether the word really has a clitic (a tag after `=` in the gloss) or the analysis is a false one.


In [ ]:
with_clitics = results[VARIANTS['all clitics']]
gained = botlikh[(botlikh['n (no clitics)'] == 0) & (botlikh['n (all clitics)'] > 0)].copy()
gained['analyses'] = gained['form'].map(lambda f: ' | '.join(with_clitics[f]))
gained['gloss'] = gained['gloss'].fillna('')
gained = (
    gained.groupby(['pos', 'form', 'gloss', 'analyses']).size().rename('frequency').reset_index()
    .sort_values(['frequency', 'pos', 'form', 'gloss'], ascending=[False, True, True, True])
)
print(gained['form'].nunique(), 'types,', int(gained['frequency'].sum()), 'tokens are recognised only with clitics')
gained.head(30)


#### 10. Statistics by part of speech

For every part of speech of the corpus and for both versions of the transducer the table `pos_stats` gives

* `words`, `recognised`, `not recognised` and `coverage`;
* `analyses`: the total number of analyses that the words of this part of speech get (not normalised);
* `analyses per word`: the same number divided by the number of all words of this part of speech (normalised);
* `analyses per recognised word`: the same number divided by the number of recognised words only.


In [ ]:
order = botlikh['pos'].value_counts().index.tolist()

rows = []
for p in order:
    part = botlikh[botlikh['pos'] == p]
    for name in VARIANTS:
        n = part['n (' + name + ')']
        rows.append({
            'pos': p, 'transducer': name, 'words': len(part),
            'recognised': int((n > 0).sum()), 'not recognised': int((n == 0).sum()),
            'coverage': round((n > 0).mean(), 3),
            'analyses': int(n.sum()),
            'analyses per word': round(n.mean(), 2),
            'analyses per recognised word': round(n[n > 0].mean(), 2) if (n > 0).any() else 0,
        })
pos_stats = pd.DataFrame(rows)
pos_stats.pivot(index='pos', columns='transducer').loc[order, [
    ('words', 'no clitics'), ('recognised', 'no clitics'), ('recognised', 'all clitics'),
    ('not recognised', 'no clitics'), ('not recognised', 'all clitics'),
    ('analyses', 'no clitics'), ('analyses', 'all clitics'),
    ('analyses per word', 'no clitics'), ('analyses per word', 'all clitics'),
    ('analyses per recognised word', 'no clitics'), ('analyses per recognised word', 'all clitics'),
]]


The figure shows four of these numbers; the number of words of every part of speech is given in brackets. In the first panel the grey bar is the number of all words, so the distance between a coloured bar and the end of the grey bar is the number of words that are not recognised; the second panel shows this number directly. The third panel is normalised by the number of words, the fourth one is not.


In [ ]:
totals = botlikh['pos'].value_counts().loc[order]
pos_labels = [p + ' (' + str(totals[p]) + ')' for p in order]


def pos_table(column):
    table = pos_stats.pivot(index='pos', columns='transducer', values=column).loc[order, list(VARIANTS)]
    table.index = pos_labels
    return table


fig, axes = plt.subplots(2, 2, figsize=(11.5, 9), constrained_layout=True)

grouped_bars(axes[0, 0], pos_table('recognised'), horizontal=True, total=totals)
finish(axes[0, 0], 'Recognised words', 'tokens (grey: all words of this part of speech)', horizontal=True, legend=True)

grouped_bars(axes[0, 1], pos_table('not recognised'), horizontal=True)
finish(axes[0, 1], 'Words without an analysis', 'tokens', horizontal=True)

grouped_bars(axes[1, 0], pos_table('analyses per word'), horizontal=True, fmt='{:.2f}')
finish(axes[1, 0], 'Analyses per word (normalised)', 'analyses / all words of this part of speech', horizontal=True)

grouped_bars(axes[1, 1], pos_table('analyses'), horizontal=True)
finish(axes[1, 1], 'Number of analyses (not normalised)', 'analyses', horizontal=True)

save_figure(fig, 'corpus_fig_pos')
plt.show()


The words that are not recognised by the transducer with clitics, with their part of speech and frequency (the most frequent ones first). This list shows what is missing in the lexicon and in the rules.


In [ ]:
not_recognised = (
    botlikh[botlikh['n (all clitics)'] == 0]
    .groupby(['pos', 'form']).agg(frequency=('word', 'size'), gloss=('gloss', 'first')).reset_index()
    .sort_values(['frequency', 'pos', 'form'], ascending=[False, True, True])
)
print(len(not_recognised), 'types,', int(not_recognised['frequency'].sum()), 'tokens are not recognised')
print(not_recognised.groupby('pos')['frequency'].agg(['size', 'sum']).rename(columns={'size': 'types', 'sum': 'tokens'})
      .sort_values('tokens', ascending=False))
not_recognised.head(30)


#### 11. Proposed part of speech and the part of speech of the corpus

Each row of the heat map is a part of speech of the corpus, each column a part of speech proposed by the transducer. A cell is the number of tokens of the row that have at least one analysis with the part of speech of the column; the last column counts the tokens without any analysis. The colour is the share of the row, so rows with few and with many words can be compared. A token with analyses of several parts of speech is counted in several columns. Cells outside the expected pairs (for example, a verb with a noun analysis) point to analyses that have to be checked.


In [ ]:
def confusion(name):
    counts = {}
    for gold, proposed_list, n in zip(botlikh['pos'], botlikh['pos (' + name + ')'], botlikh['n (' + name + ')']):
        for p in (proposed_list if n > 0 else ['no analysis']):
            counts[(gold, p)] = counts.get((gold, p), 0) + 1
    table = pd.Series(counts).unstack(fill_value=0)
    cols = [c for c in ['N', 'ADJ', 'NUM', 'PRON', 'ADV', 'POSTP', 'PTCL', 'CONJ', 'INTJ', 'ONOM', 'CLITIC'] if c in table.columns]
    return table.reindex(index=order, columns=cols + ['no analysis'], fill_value=0)


confusions = {name: confusion(name) for name in VARIANTS}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True, sharey=True)
for ax, name in zip(axes, VARIANTS):
    table = confusions[name]
    share = table.div(totals, axis=0).values
    image = ax.imshow(share, cmap=BLUES, vmin=0, vmax=1, aspect='auto')
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = table.values[i, j]
            if v:
                ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                        color='white' if share[i, j] > 0.55 else INK)
    ax.set_xticks(range(table.shape[1]))
    ax.set_xticklabels(table.columns, rotation=45, ha='right')
    ax.set_yticks(range(table.shape[0]))
    ax.set_yticklabels(pos_labels)
    ax.set_title(name, pad=10)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
axes[0].set_ylabel('part of speech in the corpus (tokens)')
fig.supxlabel('part of speech proposed by the transducer', fontsize=10, color=INK2)
bar = fig.colorbar(image, ax=axes, shrink=0.7, pad=0.015, ticks=[0, 0.25, 0.5, 0.75, 1])
bar.set_label('share of the tokens of the row')
bar.outline.set_visible(False)
bar.ax.tick_params(length=0)

save_figure(fig, 'corpus_fig_proposed_pos')
plt.show()
confusions['all clitics']


#### 12. Saving the results

A cell that begins with `=` (the analysis of an isolated clitic, `=<quot>`, or a segmentation such as `=талу`) would be read by Excel as a formula, so such cells are stored as text. The four tables for the manual evaluation go to one Excel file (with the name of the analyzer in the file name) and to text files. The statistics are saved as text files, the figures as PNG and PDF. The checksums of the text files are printed for comparison with another run.


In [ ]:
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

WIDTH = {'analyses': 60, 'analysis': 60, 'sentence': 45, 'translation_ru': 45, 'translation_en': 45,
         'gloss': 28, 'glosses': 40, 'segmentation': 22, 'document': 28, 'comment': 30}

xlsx = os.path.join(OUT_DIR, 'corpus_words_' + TABLE_ANALYZER + '.xlsx')
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    for sheet, df in {'annotation': table_annotation, 'words': table_words, 'analyses': table_analyses, 'types': table_types}.items():
        df.to_excel(xw, sheet_name=sheet, index=False)
        ws = xw.sheets[sheet]
        ws.freeze_panes = 'A2'
        for k, col in enumerate(df.columns, 1):
            ws.column_dimensions[get_column_letter(k)].width = WIDTH.get(str(col), 16)
            ws.cell(row=1, column=k).font = Font(bold=True)
        for row in ws.iter_rows(min_row=2):
            for c in row:
                if c.data_type == 'f':
                    c.data_type = 's'
                c.alignment = Alignment(wrap_text=True, vertical='top')

tsv = {
    'corpus_annotation_' + TABLE_ANALYZER + '.tsv': table_annotation,
    'corpus_analyses_' + TABLE_ANALYZER + '.tsv': table_analyses,
    'corpus_types_' + TABLE_ANALYZER + '.tsv': table_types,
    'corpus_metrics.tsv': metrics_table,
    'corpus_pos_stats.tsv': pos_stats,
    'corpus_not_recognised.tsv': not_recognised,
    'corpus_recognised_only_with_clitics.tsv': gained,
    'corpus_clitics.tsv': clitics,
    'corpus_proposed_pos_no_clitics.tsv': confusions['no clitics'].reset_index(),
    'corpus_proposed_pos_all_clitics.tsv': confusions['all clitics'].reset_index(),
}
for name, df in tsv.items():
    df.to_csv(os.path.join(OUT_DIR, name), sep='\t', index=False)
    print(sha256(os.path.join(OUT_DIR, name)), name)
print('written:', sorted(os.listdir(OUT_DIR)))


All tables and figures are in `OUT_DIR`. In Colab the folder is packed into one archive and downloaded.


In [ ]:
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
try:
    from google.colab import files
    files.download(OUT_DIR + '.zip')
except ImportError:
    print('results:', os.path.abspath(OUT_DIR))
